<a href="https://colab.research.google.com/github/mitalidaduria/enterprise-data-platform/blob/main/Big_Data_Engineering_%26_PySpark_Ingestion!.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/mitalidaduria/enterprise-data-platform.git
%cd enterprise-data-platform

Cloning into 'enterprise-data-platform'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 23 (delta 5), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 7.76 KiB | 7.76 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/enterprise-data-platform


In [2]:
!git config --global user.name "Mitali Daduria"
!git config --global user.email "mitalidaduriaj@gmail.com"

In [3]:
import os
os.makedirs("pipelines", exist_ok=True)

**Synthetic Data Generator:**

In [4]:
%%writefile pipelines/generate_data.py
import os
import csv
import uuid
import hashlib
import random

DATA_DIR = os.path.join(os.path.dirname(__file__), "raw_data")
os.makedirs(DATA_DIR, exist_ok=True)

NUM_RECORD_PAIRS = 1000

FIRST_NAMES = ["Robert", "Bob", "Rob", "William", "Bill", "Elizabeth", "Liz", "Michael", "Mike", "Sarah"]
LAST_NAMES = ["Jones", "Smith", "Taylor", "Brown", "Wilson", "Davies", "Evans", "Thomas", "Johnson"]
STREETS = ["123 Main St", "456 Oak Ave", "789 Pine Rd", "101 Maple Dr", "202 Birch Ln"]
DOMAINS = ["gmail.com", "yahoo.com", "hotmail.com", "enterprise.org"]

def generate_datasets():
    billing_rows = []
    shipping_rows = []

    for i in range(NUM_RECORD_PAIRS):
        first_name = random.choice(FIRST_NAMES)
        last_name = random.choice(LAST_NAMES)
        full_name = f"{first_name} {last_name}"
        domain = random.choice(DOMAINS)

        raw_email = f"{first_name.lower()}.{last_name.lower()}{i}@{domain}"

        billing_id = f"BIL-{uuid.uuid4().hex[:8].upper()}"
        credit_card_hash = hashlib.sha256(f"CARD-{i}".encode()).hexdigest()
        billing_address = random.choice(STREETS)

        billing_rows.append([billing_id, full_name, raw_email, credit_card_hash, billing_address])

        shipping_id = f"SHP-{uuid.uuid4().hex[:8].upper()}"
        recipient_name = f"{first_name[0]}. {last_name}"
        phone_number = f"555-{random.randint(100, 999)}-{random.randint(1000, 9999)}"
        street_address = billing_address

        shipping_rows.append([shipping_id, recipient_name, raw_email, street_address, phone_number])

    billing_path = os.path.join(DATA_DIR, "raw_billing.csv")
    with open(billing_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["billing_id", "full_name", "email", "credit_card_hash", "billing_address"])
        writer.writerows(billing_rows)

    shipping_path = os.path.join(DATA_DIR, "raw_shipping.csv")
    with open(shipping_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["shipping_id", "recipient_name", "email", "street_address", "phone_number"])
        writer.writerows(shipping_rows)

    print(f"✅ Generated {NUM_RECORD_PAIRS} synthetic raw records in '{DATA_DIR}'")

if __name__ == "__main__":
    generate_datasets()

Writing pipelines/generate_data.py


**Ingestion Pipeline:**

In [5]:
%%writefile pipelines/ingest_pyspark.py
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, sha2, col

def run_ingestion_pipeline():
    spark = SparkSession.builder \
        .appName("EnterpriseDataIngestion") \
        .master("local[*]") \
        .getOrCreate()

    spark.sparkContext.setLogLevel("ERROR")
    print("🚀 PySpark Session Initialized Successfully.")

    base_dir = os.path.dirname(__file__)
    raw_dir = os.path.join(base_dir, "raw_data")
    staging_dir = os.path.join(base_dir, "staging_data")
    os.makedirs(staging_dir, exist_ok=True)

    billing_raw_path = os.path.join(raw_dir, "raw_billing.csv")
    if os.path.exists(billing_raw_path):
        df_billing = spark.read.csv(billing_raw_path, header=True, inferSchema=True)
        df_billing_transformed = df_billing \
            .withColumn("email_hash", sha2(col("email"), 256)) \
            .withColumn("src_system_id", lit("BILLING_DB")) \
            .withColumn("ingested_at", current_timestamp()) \
            .withColumn("pipeline_version", lit("v1.0.0"))

        billing_out = os.path.join(staging_dir, "stg_billing.parquet")
        df_billing_transformed.write.mode("overwrite").parquet(billing_out)
        print(f"✅ Billing Data Ingested & Cleansed: {df_billing_transformed.count()} rows -> Parquet")

    shipping_raw_path = os.path.join(raw_dir, "raw_shipping.csv")
    if os.path.exists(shipping_raw_path):
        df_shipping = spark.read.csv(shipping_raw_path, header=True, inferSchema=True)
        df_shipping_transformed = df_shipping \
            .withColumn("email_hash", sha2(col("email"), 256)) \
            .withColumn("src_system_id", lit("SHIPPING_DB")) \
            .withColumn("ingested_at", current_timestamp()) \
            .withColumn("pipeline_version", lit("v1.0.0"))

        shipping_out = os.path.join(staging_dir, "stg_shipping.parquet")
        df_shipping_transformed.write.mode("overwrite").parquet(shipping_out)
        print(f"✅ Shipping Data Ingested & Cleansed: {df_shipping_transformed.count()} rows -> Parquet")

    spark.stop()

if __name__ == "__main__":
    run_ingestion_pipeline()

Writing pipelines/ingest_pyspark.py


In [6]:
!pip install pyspark -q
!python pipelines/generate_data.py
!python pipelines/ingest_pyspark.py

✅ Generated 1000 synthetic raw records in '/content/enterprise-data-platform/pipelines/raw_data'
[0.031s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.031s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/09 13:51:33 WARN NativeCodeL

**Data Quality Gates:**

In [7]:
%%writefile pipelines/data_quality.py
"""
Enterprise Data Platform: PySpark Data Quality Gates & Quarantine Pipeline
Validates staging datasets against strict domain rules and segregates invalid records.
"""

import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, current_timestamp, lit

def run_quality_gates():
    # 1. Initialize PySpark Session
    spark = SparkSession.builder \
        .appName("EnterpriseDataQualityGates") \
        .master("local[*]") \
        .getOrCreate()

    spark.sparkContext.setLogLevel("ERROR")
    print("🛡️ Initializing Data Quality Validation Engine...")

    base_dir = os.path.dirname(__file__)
    staging_dir = os.path.join(base_dir, "staging_data")
    validated_dir = os.path.join(base_dir, "validated_data")
    quarantine_dir = os.path.join(base_dir, "quarantine_data")
    os.makedirs(validated_dir, exist_ok=True)
    os.makedirs(quarantine_dir, exist_ok=True)

    # ---------------------------------------------------------------------------
    # 2. Validate Billing Data
    # ---------------------------------------------------------------------------
    billing_path = os.path.join(staging_dir, "stg_billing.parquet")
    if os.path.exists(billing_path):
        df_billing = spark.read.parquet(billing_path)
        total_count = df_billing.count()

        # Quality Rule: Primary key must not be null/empty AND email_hash must be exactly 64 chars (SHA-256)
        rule_valid_pk = col("billing_id").isNotNull() & (col("billing_id") != "")
        rule_valid_hash = col("email_hash").isNotNull() & (length(col("email_hash")) == 64)
        valid_condition = rule_valid_pk & rule_valid_hash

        df_valid = df_billing.filter(valid_condition)
        df_quarantine = df_billing.filter(~valid_condition) \
            .withColumn("quarantine_reason", lit("Failed PK or SHA-256 Hash Length Check")) \
            .withColumn("quarantined_at", current_timestamp())

        # Write outputs
        df_valid.write.mode("overwrite").parquet(os.path.join(validated_dir, "stg_billing_validated.parquet"))
        if df_quarantine.count() > 0:
            df_quarantine.write.mode("overwrite").parquet(os.path.join(quarantine_dir, "stg_billing_quarantine.parquet"))

        print(f"✅ [Billing Gate] Total: {total_count} | Valid: {df_valid.count()} | Quarantined: {df_quarantine.count()}")

    # ---------------------------------------------------------------------------
    # 3. Validate Shipping Data
    # ---------------------------------------------------------------------------
    shipping_path = os.path.join(staging_dir, "stg_shipping.parquet")
    if os.path.exists(shipping_path):
        df_shipping = spark.read.parquet(shipping_path)
        total_count_s = df_shipping.count()

        # Quality Rule: PK, Hash length, and Phone number must be present
        rule_valid_pk = col("shipping_id").isNotNull() & (col("shipping_id") != "")
        rule_valid_hash = col("email_hash").isNotNull() & (length(col("email_hash")) == 64)
        rule_valid_phone = col("phone_number").isNotNull()
        valid_condition_s = rule_valid_pk & rule_valid_hash & rule_valid_phone

        df_valid_s = df_shipping.filter(valid_condition_s)
        df_quarantine_s = df_shipping.filter(~valid_condition_s) \
            .withColumn("quarantine_reason", lit("Failed PK, Hash, or Phone Check")) \
            .withColumn("quarantined_at", current_timestamp())

        # Write outputs
        df_valid_s.write.mode("overwrite").parquet(os.path.join(validated_dir, "stg_shipping_validated.parquet"))
        if df_quarantine_s.count() > 0:
            df_quarantine_s.write.mode("overwrite").parquet(os.path.join(quarantine_dir, "stg_shipping_quarantine.parquet"))

        print(f"✅ [Shipping Gate] Total: {total_count_s} | Valid: {df_valid_s.count()} | Quarantined: {df_quarantine_s.count()}")

    spark.stop()

if __name__ == "__main__":
    run_quality_gates()

Writing pipelines/data_quality.py


In [8]:
!python pipelines/data_quality.py

[0.006s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.006s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/09 15:42:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where

**Entity Resolution Pipeline:**

In [9]:
%%writefile pipelines/entity_resolution.py
"""
Enterprise Data Platform: PySpark Entity Resolution & MDM Engine
Executes deterministic matching on PII hashes and applies Survivorship Rules
to produce the authoritative Customer Golden Record dataset.
"""

import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, coalesce, when, lit, current_timestamp, sha2, concat_ws
)

def run_entity_resolution():
    # 1. Initialize PySpark Session
    spark = SparkSession.builder \
        .appName("EnterpriseMDMEntityResolution") \
        .master("local[*]") \
        .getOrCreate()

    spark.sparkContext.setLogLevel("ERROR")
    print("🔄 Starting Master Data Management (MDM) Entity Resolution...")

    base_dir = os.path.dirname(__file__)
    validated_dir = os.path.join(base_dir, "validated_data")

    billing_path = os.path.join(validated_dir, "stg_billing_validated.parquet")
    shipping_path = os.path.join(validated_dir, "stg_shipping_validated.parquet")

    if not (os.path.exists(billing_path) and os.path.exists(shipping_path)):
        print("❌ Validated datasets missing. Run data_quality.py first.")
        spark.stop()
        return

    # 2. Read Validated Staging Parquet Files
    df_billing = spark.read.parquet(billing_path).alias("bil")
    df_shipping = spark.read.parquet(shipping_path).alias("shp")

    # 3. Deterministic Matching via FULL OUTER JOIN on email_hash
    df_matched = df_billing.join(
        df_shipping,
        col("bil.email_hash") == col("shp.email_hash"),
        "full_outer"
    )

    # 4. Apply Survivorship Rules & Construct Golden Record
    anchor_email_hash = coalesce(col("bil.email_hash"), col("shp.email_hash"))

    df_golden = df_matched.select(
        sha2(concat_ws("_", lit("GOLDEN"), anchor_email_hash), 256).alias("golden_customer_id"),
        coalesce(col("bil.full_name"), col("shp.recipient_name")).alias("primary_name"),
        anchor_email_hash.alias("primary_email_hash"),
        col("shp.phone_number").alias("primary_phone"),
        coalesce(col("shp.street_address"), col("bil.billing_address")).alias("primary_address"),
        col("bil.billing_id").alias("billing_linkage_id"),
        col("shp.shipping_id").alias("shipping_linkage_id"),
        when(col("bil.billing_id").isNotNull() & col("shp.shipping_id").isNotNull(), 2)
        .otherwise(1).alias("total_source_linkages"),
        current_timestamp().alias("created_at"),
        current_timestamp().alias("updated_at"),
        lit("v1.0.0").alias("pipeline_version")
    )

    # 5. Persist Golden Record to Parquet
    golden_out = os.path.join(validated_dir, "dim_customer_golden.parquet")
    df_golden.write.mode("overwrite").parquet(golden_out)

    total_golden = df_golden.count()
    multi_source_matches = df_golden.filter(col("total_source_linkages") == 2).count()

    print(f"✨ Entity Resolution Complete!")
    print(f"📊 Total Golden Records: {total_golden}")
    print(f"🔗 Successfully Merged Multi-Source Entities: {multi_source_matches}")

    spark.stop()

if __name__ == "__main__":
    run_entity_resolution()

Writing pipelines/entity_resolution.py


In [10]:
!python pipelines/entity_resolution.py

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/09 15:58:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where

In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("VerifyMDM").master("local[*]").getOrCreate()

# Load the Golden Record Parquet file generated by Day 7
golden_df = spark.read.parquet("pipelines/validated_data/dim_customer_golden.parquet")

print("--- SAMPLE GOLDEN RECORDS ---")
golden_df.select("golden_customer_id", "primary_name", "primary_email_hash", "total_source_linkages").show(5, truncate=False)

total_records = golden_df.count()
multi_source = golden_df.filter(golden_df.total_source_linkages == 2).count()

print(f"📊 Total Unique Golden Records: {total_records}")
print(f"🔗 Successfully Merged Across Both Systems (Billing + Shipping): {multi_source}")

spark.stop()

--- SAMPLE GOLDEN RECORDS ---
+----------------------------------------------------------------+------------+----------------------------------------------------------------+---------------------+
|golden_customer_id                                              |primary_name|primary_email_hash                                              |total_source_linkages|
+----------------------------------------------------------------+------------+----------------------------------------------------------------+---------------------+
|f3fa1d89435ecfa92a4d95549c7b8d9ba50df6bcbb25ae712a5946586de3a835|Mike Davies |003b2bd2c92bd367ee87f8c18794d4486451707b0f55b3d3ad2d5a552b651199|2                    |
|42680d1ac8e36fc7f7de6391c8147482a6aaf995338ee46574c8f781023e3cee|Bob Thomas  |00c00382d0cc25a71b4b44d661d9abd7247a093028c688cd50b5d7346539ecc2|2                    |
|424a2663e38a67fc4ffdca391cbcabc778240fc344fe7ad108ae7e85b00d6290|Rob Jones   |01cb93ded7e41c42146033ef7cc2bb2318b75b9b31062cd91e8f586a